<small>DAY 51 / SPARK IN DOCKER / 01</small>

# Build a Spark image

**Ubuntu + Java 11 + Spark 3.5.9**

Build one image. Start a master and worker with `docker run`. Run the same setup with Docker Compose.

The files are in **`spark-docker/`**, beside this notebook. Complete file contents are included below for copying.


<small>DAY 51 / SPARK IN DOCKER / 02</small>

# Where to run this

Use the **remote Ubuntu VM only**, not WSL or a corporate laptop. Docker and Compose should already be installed using the preceding notebook.

Run the commands in the VM terminal. The notebook contains instructions, not executable Python setup cells.

Allow roughly **4 CPUs, 8 GB RAM, and 10 GB free disk**. Downloads need internet access. Java and Spark are installed inside the image.


<small>DAY 51 / SPARK IN DOCKER / 03</small>

# The folder to upload

```text
Day51/
  D511_Spark_Docker_Image_and_Compose.ipynb
  spark-docker/
    Dockerfile
    entrypoint.sh
    download-jars.sh
    compose.yaml
    README.md
    .dockerignore
    .gitignore
    .gitattributes
    jars/                 # Created by the downloader; not committed
```

Upload the text files to GitHub. Anyone can clone or download them and follow `README.md`. One Dockerfile serves both roles.


<small>DAY 51 / SPARK IN DOCKER / 04</small>

# One image, two roles

The **master** allocates cluster resources. The **worker** launches executors that perform the work.

We submit from the master container for convenience. That creates a separate **driver process** there; the master process is not the driver.




<small>DAY 51 / SPARK IN DOCKER / 05</small>

# Keep versions aligned

| Component | Choice |
|---|---|
| Base | Ubuntu with `apt` packages |
| Java | OpenJDK 11 |
| Spark | 3.5.9, Hadoop 3 distribution |
| Scala connectors | `_2.12` |
| Python | Ubuntu's `python3` in both roles |

[Spark 3.5.9 supports Java 11](https://spark.apache.org/docs/3.5.9/). Connector names must match the Spark and Scala versions. There is no separate host Python virtual environment inside these containers.


<small>DAY 51 / SPARK IN DOCKER / 06</small>

# Download the connector JARs

From the course repository root on the VM:

```bash
cd Day51/spark-docker
sudo apt update
sudo apt install -y curl ca-certificates
bash download-jars.sh
ls -lh jars/
```

If you downloaded `spark-docker` as its own repository, open that folder instead.
The next cell contains the complete downloader. Save it as `download-jars.sh` if recreating the files manually.


<small>DAY 51 / SPARK IN DOCKER / 07</small>

# File: download-jars.sh

```bash
#!/usr/bin/env bash
set -euo pipefail
cd "$(dirname "${BASH_SOURCE[0]}")"

case "${1:-}" in
  ""|--with-s3) ;;
  *) echo "Usage: bash download-jars.sh [--with-s3]" >&2; exit 2 ;;
esac
mkdir -p jars
base=https://repo.maven.apache.org/maven2

fetch() {
  local path="$1"
  local file="jars/${path##*/}"
  echo "Downloading ${file##*/}"
  curl --fail --location --retry 3 --connect-timeout 20 \
    --output "$file.part" "$base/$path"
  mv "$file.part" "$file"
}

# Spark connectors match Spark 3.5.9 and Scala 2.12.
fetch io/delta/delta-spark_2.12/3.3.2/delta-spark_2.12-3.3.2.jar
fetch io/delta/delta-storage/3.3.2/delta-storage-3.3.2.jar
fetch org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.10.0/iceberg-spark-runtime-3.5_2.12-1.10.0.jar
fetch org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.9/spark-sql-kafka-0-10_2.12-3.5.9.jar
fetch org/apache/spark/spark-token-provider-kafka-0-10_2.12/3.5.9/spark-token-provider-kafka-0-10_2.12-3.5.9.jar
fetch org/apache/kafka/kafka-clients/3.4.1/kafka-clients-3.4.1.jar
fetch org/apache/commons/commons-pool2/2.11.1/commons-pool2-2.11.1.jar
fetch org/apache/spark/spark-avro_2.12/3.5.9/spark-avro_2.12-3.5.9.jar
fetch com/mysql/mysql-connector-j/8.4.0/mysql-connector-j-8.4.0.jar
fetch org/postgresql/postgresql/42.7.8/postgresql-42.7.8.jar

# Optional large bundles: Iceberg S3FileIO and Hadoop S3A.
if [[ "${1:-}" == "--with-s3" ]]; then
  fetch org/apache/iceberg/iceberg-aws-bundle/1.10.0/iceberg-aws-bundle-1.10.0.jar
  fetch org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar
  fetch com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar
fi

echo "JARs ready in $(pwd)/jars. Rebuild the image after changing JARs."
```


<small>DAY 51 / SPARK IN DOCKER / 08</small>

# What the JARs provide

| Feature | JAR versions |
|---|---|
| Delta Lake | Spark and storage 3.3.2 |
| Iceberg | Runtime 1.10.0 for Spark 3.5 / Scala 2.12 |
| Kafka | SQL + token-provider 3.5.9; client 3.4.1; pool 2.11.1 |
| Avro | Spark Avro 3.5.9 |
| JDBC | MySQL 8.4.0; PostgreSQL 42.7.8 |

The Kafka token-provider is included as a dependency. Spark's distribution supplies the base runtime libraries.

[Delta compatibility](https://docs.delta.io/releases/) ? [Spark Kafka integration](https://spark.apache.org/docs/3.5.9/structured-streaming-kafka-integration.html)


<small>DAY 51 / SPARK IN DOCKER / 09</small>

# File: Dockerfile

```dockerfile
FROM ubuntu:22.04

RUN apt-get update && apt-get install -y --no-install-recommends \
    openjdk-11-jdk-headless python3 curl ca-certificates procps bash \
    && ln -s "$(dirname "$(dirname "$(readlink -f "$(command -v javac)")")")" /opt/java \
    && rm -rf /var/lib/apt/lists/*

ENV JAVA_HOME=/opt/java \
    SPARK_HOME=/opt/spark \
    PYSPARK_PYTHON=/usr/bin/python3 \
    PYSPARK_DRIVER_PYTHON=/usr/bin/python3
ENV PATH="${SPARK_HOME}/bin:${JAVA_HOME}/bin:${PATH}"

# Spark 3.5.9, Hadoop 3 client libraries, Scala 2.12.
RUN cd /tmp \
    && curl -fL --retry 3 -O https://archive.apache.org/dist/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz \
    && curl -fL --retry 3 -O https://archive.apache.org/dist/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz.sha512 \
    && sha512sum -c spark-3.5.9-bin-hadoop3.tgz.sha512 \
    && tar -xzf spark-3.5.9-bin-hadoop3.tgz -C /opt \
    && mv /opt/spark-3.5.9-bin-hadoop3 /opt/spark \
    && rm spark-3.5.9-bin-hadoop3.tgz spark-3.5.9-bin-hadoop3.tgz.sha512

# Run bash download-jars.sh on the VM before building.
COPY jars/*.jar /opt/spark/jars/
COPY entrypoint.sh /usr/local/bin/spark-entrypoint.sh
RUN sed -i 's/\r$//' /usr/local/bin/spark-entrypoint.sh \
    && chmod +x /usr/local/bin/spark-entrypoint.sh \
    && mkdir -p /opt/spark/work /opt/spark/data

WORKDIR /opt/spark
EXPOSE 7077 7078 8080 8081 4040
ENTRYPOINT ["/usr/local/bin/spark-entrypoint.sh"]
CMD ["master"]
```


<small>DAY 51 / SPARK IN DOCKER / 10</small>

# Read the build in five steps

1. Start from Ubuntu and install Java 11, Python, and download tools.
2. Set `JAVA_HOME`, `SPARK_HOME`, and the Python executable.
3. Download Spark, verify its checksum, and extract it.
4. Copy local connector JARs into Spark's `jars` directory.
5. Use the startup script to choose a master, worker, or utility command.

`COPY jars/*.jar /opt/spark/jars/` is a **build-time copy**. Download or change JARs first, then rebuild. Both services receive the same connector files.


<small>DAY 51 / SPARK IN DOCKER / 11</small>

# File: entrypoint.sh

```bash
#!/usr/bin/env bash
set -euo pipefail

# Run Spark in the foreground so the container follows its lifetime.
case "${1:-master}" in
  master)
    exec "$SPARK_HOME/bin/spark-class" org.apache.spark.deploy.master.Master \
      --host "${SPARK_MASTER_HOST:-spark-master}" \
      --port 7077 --webui-port 8080
    ;;
  worker)
    exec "$SPARK_HOME/bin/spark-class" org.apache.spark.deploy.worker.Worker \
      --host "${SPARK_WORKER_HOST:-$(hostname)}" \
      --port 7078 --webui-port 8081 \
      --cores "${SPARK_WORKER_CORES:-2}" \
      --memory "${SPARK_WORKER_MEMORY:-2g}" \
      "${SPARK_MASTER_URL:-spark://spark-master:7077}"
    ;;
  *)
    # Also supports: docker run --rm IMAGE java -version
    exec "$@"
    ;;
esac
```


<small>DAY 51 / SPARK IN DOCKER / 12</small>

# Keep the process in the foreground

The startup script uses `exec` to run Spark's main class in the foreground. Docker can track its lifetime and send stop signals to it.

The usual `start-master.sh` and `start-worker.sh` scripts daemonize by default. A container can exit when its startup script returns.

The default command is `master`. Passing `worker` changes the role; passing `java -version` runs a utility command instead.


<small>DAY 51 / SPARK IN DOCKER / 13</small>

# Small Git and build files

**`.dockerignore`** keeps only the image inputs:

```text
*
!Dockerfile
!entrypoint.sh
!jars/
!jars/*.jar
```

**`.gitignore`** excludes downloaded files:

```text
jars/
*.tgz
*.part
```

**`.gitattributes`** preserves Linux line endings:

```text
*.sh text eol=lf
Dockerfile text eol=lf
*.yaml text eol=lf
```


<small>DAY 51 / SPARK IN DOCKER / 14</small>

# Build and list the image

```bash
docker build -t day51-spark:3.5.9-java11 .
docker image ls day51-spark
```

`-t` assigns the image name and tag. The final `.` points to this folder as the build context.

The first build downloads Ubuntu packages and Spark. Later builds can reuse unchanged layers.

If `COPY jars/*.jar` fails, run the JAR downloader and build again.


<small>DAY 51 / SPARK IN DOCKER / 15</small>

# Check the image before starting a cluster

```bash
docker run --rm day51-spark:3.5.9-java11 java -version
docker run --rm day51-spark:3.5.9-java11 spark-submit --version
docker run --rm day51-spark:3.5.9-java11 bash -c 'ls /opt/spark/jars/*delta* /opt/spark/jars/*kafka*'
```

Expect Java **11**, Spark **3.5.9**, and the connector filenames.
`--rm` removes each temporary container when its check finishes.


<small>DAY 51 / SPARK IN DOCKER / 16</small>

# Run the master

```bash
docker network create spark-lab
docker volume create day51-spark-data

docker run -d --init --name spark-master --hostname spark-master \
  --network spark-lab \
  -p 127.0.0.1:8080:8080 -p 127.0.0.1:4040:4040 \
  -v day51-spark-data:/opt/spark/data \
  day51-spark:3.5.9-java11 master
```

The network provides container-name lookup. The named volume keeps shared lab files outside the containers.

The master UI starts with **zero workers**. Spark port 7077 stays inside the Docker network.


<small>DAY 51 / SPARK IN DOCKER / 17</small>

# Open the Spark master UI

From your local terminal, connect to the remote VM with port forwarding:

```bash
ssh -L 8080:127.0.0.1:8080 -L 8081:127.0.0.1:8081 \
    -L 4040:127.0.0.1:4040 user@vm-address
```

Keep that connection open. Browse to **http://localhost:8080**.

The worker UI will use **8081**. A running application's UI uses **4040**.
Stop the earlier Nginx/Adminer example if it occupies these VM ports. No local Docker installation is needed.


<small>DAY 51 / SPARK IN DOCKER / 18</small>

# Run the worker

```bash
docker run -d --init --name spark-worker --hostname spark-worker \
  --network spark-lab \
  -p 127.0.0.1:8081:8081 \
  -e SPARK_MASTER_URL=spark://spark-master:7077 \
  -e SPARK_WORKER_CORES=2 -e SPARK_WORKER_MEMORY=2g \
  -v day51-spark-data:/opt/spark/data \
  day51-spark:3.5.9-java11 worker
```

Refresh the master UI: expect **one ALIVE worker**, **2 cores**, and **2 GB** available memory.

These values describe Spark scheduling capacity; they are not container resource limits.


<small>DAY 51 / SPARK IN DOCKER / 19</small>

# Confirm registration

```bash
docker ps
docker logs spark-master
docker logs spark-worker
```

Look for the worker's successful registration message. The worker must use `spark://spark-master:7077` on the same Docker network.

Do not use `localhost:7077` in the worker: that points back to the worker container.


<small>DAY 51 / SPARK IN DOCKER / 20</small>

# Submit the bundled Pi example

```bash
docker exec -it spark-master spark-submit \
  --master spark://spark-master:7077 --deploy-mode client \
  --conf spark.driver.host=spark-master \
  --conf spark.driver.bindAddress=0.0.0.0 \
  --total-executor-cores 2 --executor-memory 1g \
  /opt/spark/examples/src/main/python/pi.py 10
```

Expect **`Pi is roughly ...`** in the terminal and a completed application in the master UI.

The driver advertises `spark-master` so executors can call it back. Python uses **client** deploy mode in this standalone example.


<small>DAY 51 / SPARK IN DOCKER / 21</small>

# Move the same setup to Compose

Remove the manually started containers first so the UI ports are free:

```bash
docker stop spark-worker spark-master
docker rm spark-worker spark-master
docker network rm spark-lab
```

The next cell is the complete `compose.yaml`. Both services use the image you just built.

Compose creates a project network and a shared data volume. Its volume is separate from the manual example's volume.


<small>DAY 51 / SPARK IN DOCKER / 22</small>

# File: compose.yaml

```yaml
name: day51-spark
services:
  spark-master:
    build: .
    image: day51-spark:3.5.9-java11
    hostname: spark-master
    init: true
    command: ["master"]
    environment:
      SPARK_MASTER_HOST: spark-master
    ports:
      - "127.0.0.1:8080:8080"
      - "127.0.0.1:4040:4040"
    volumes:
      - spark-data:/opt/spark/data
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8080/"]
      interval: 5s
      timeout: 3s
      retries: 20
      start_period: 10s

  spark-worker:
    image: day51-spark:3.5.9-java11
    hostname: spark-worker
    init: true
    command: ["worker"]
    environment:
      SPARK_MASTER_URL: spark://spark-master:7077
      SPARK_WORKER_HOST: spark-worker
      SPARK_WORKER_CORES: "2"
      SPARK_WORKER_MEMORY: 2g
    ports:
      - "127.0.0.1:8081:8081"
    volumes:
      - spark-data:/opt/spark/data
    depends_on:
      spark-master:
        condition: service_healthy

volumes:
  spark-data:
```


<small>DAY 51 / SPARK IN DOCKER / 23</small>

# Start with Compose

```bash
docker compose config -q
docker compose build
docker compose up -d --pull never
docker compose ps
docker compose logs spark-worker
```

Build first: the worker reuses the image built by the master service definition.

The worker starts after the master's UI responds to its health check. Refresh **localhost:8080** and confirm that the worker is registered.


<small>DAY 51 / SPARK IN DOCKER / 24</small>

# Submit with Compose

```bash
docker compose exec spark-master spark-submit \
  --master spark://spark-master:7077 --deploy-mode client \
  --conf spark.driver.host=spark-master \
  --conf spark.driver.bindAddress=0.0.0.0 \
  --total-executor-cores 2 --executor-memory 1g \
  /opt/spark/examples/src/main/python/pi.py 10
```

The command after `spark-master` runs inside that service's container. Compose resolves the actual container name for you.


<small>DAY 51 / SPARK IN DOCKER / 25</small>

# Keep an application open

```bash
docker compose exec spark-master pyspark \
  --master spark://spark-master:7077 \
  --conf spark.driver.host=spark-master \
  --conf spark.driver.bindAddress=0.0.0.0 \
  --total-executor-cores 2 --executor-memory 1g
```

At the Python prompt:

```python
spark.range(1, 11).show()
print(spark.sparkContext.master)
```

View **http://localhost:4040** while the shell stays open. Finish with `spark.stop()` and `exit()`. The application UI closes when the driver exits.


<small>DAY 51 / SPARK IN DOCKER / 26</small>

# Shared files and connector settings

Both services mount the same volume at **`/opt/spark/data`**. Use that path for shared local input and output on this VM.

The Hadoop-enabled Spark download includes client libraries. It does not start HDFS or YARN.

Copied JARs provide classes. Delta/Iceberg catalog settings, Kafka brokers, JDBC databases, and credentials still need their own configuration. Delta's Python helper package is not installed by copying a JAR.


<small>DAY 51 / SPARK IN DOCKER / 27</small>

# Optional AWS and S3 JARs

```bash
bash download-jars.sh --with-s3
docker compose build
docker compose up -d --force-recreate --pull never
```

Adds Iceberg AWS bundle **1.10.0**, Hadoop AWS **3.3.4**, and AWS SDK bundle **1.12.262**.

These larger downloads are optional. No credentials are baked into the image. Running the downloader without the option later does not remove existing JARs.

[Matching Spark dependency versions](https://github.com/apache/spark/blob/v3.5.9/pom.xml)


<small>DAY 51 / SPARK IN DOCKER / 28</small>

# Stop and clean up

```bash
docker compose down
```

Stops and removes the Compose containers and network. The shared data volume remains.

To deliberately remove that data too:

```bash
docker compose down -v
```

The manual example's volume is separate. After removing its containers, delete it only if no longer needed:

```bash
docker volume rm day51-spark-data
```


<small>DAY 51 / SPARK IN DOCKER / 29</small>

# Ready to share

Upload `spark-docker/` and this notebook to GitHub. Keep the downloaded `jars/` directory out of Git; the script recreates it.

From a fresh download, the shortest path is:

```bash
bash download-jars.sh
docker compose build
docker compose up -d --pull never
```

Run those commands from `spark-docker` after installing Docker and the host download tools. Then open the forwarded master UI and submit the sample job.

The directory's `README.md` includes both the manual and Compose workflows.


<small>DAY 51 / SPARK IN DOCKER / 30</small>

# When something does not start

| Symptom | Check |
|---|---|
| Missing JARs at build time | Run `bash download-jars.sh` |
| Worker absent from UI | Check logs, master URL, and common network |
| Executor cannot reach driver | Use the provided driver host settings |
| Port already in use | Stop the previous exercise or adjust forwarding and port mappings |
| Script reports `bad interpreter` | Save shell scripts with LF endings |
| Job waits for resources | Ensure the worker is registered and offers enough memory |

[Spark standalone reference](https://spark.apache.org/docs/3.5.9/spark-standalone.html)
